# VitalSync: Patient Physiological Vitals Forecasting with LSTM
## Phase 2: Multi-Horizon Forecasting & Uncertainty Quantification

This notebook mirrors the training and evaluation pipeline implemented in `ml/train.py`:
1. **Strict Patient-Level Partitioning** (70% Train, 15% Val, 15% Test) — Zero cross-patient leakage.
2. **Feature Normalization** — StandardScaler fit on training cohort only.
3. **Sliding Window Construction** — 60-minute lookback ($T_{in}=60$), 12-minute forward horizon ($T_{out}=12$).
4. **Clinical Baseline Benchmarking** — Persistence (Last Value) and 15-minute Moving Average.
5. **Deep Recurrent Modeling** — Stacked LSTM with Dropout regularization.
6. **Real Clinical Metric Evaluation** — MAE & RMSE in BPM and mmHg per step and feature.
7. **Predictive Uncertainty** — Monte Carlo Dropout (30 stochastic passes, 90% confidence bands).

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# Set project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT_DIR)

print("TensorFlow Version:", tf.__version__)
print("Project Root:", ROOT_DIR)

## 1. Load Processed Dataset & Patient Partitioning

In [ ]:
data_path = os.path.join(ROOT_DIR, "ml", "data", "processed", "vitals.parquet")
df = pd.read_parquet(data_path)
print(f"Total Observations: {len(df):,}")
print(f"Total Unique Patients: {df['patient_id'].nunique()}")
df.head()

In [ ]:
from ml.train import load_and_preprocess_data, build_sliding_windows, FEATURES, LOOKBACK, HORIZON

train_df, val_df, test_df, scaler, p_train, p_val, p_test = load_and_preprocess_data(data_path)

print("Generating sliding windows (60 min lookback -> 12 min horizon)...")
X_train, Y_train = build_sliding_windows(train_df, scaler, LOOKBACK, HORIZON)
X_val, Y_val = build_sliding_windows(val_df, scaler, LOOKBACK, HORIZON)
X_test, Y_test = build_sliding_windows(test_df, scaler, LOOKBACK, HORIZON)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape:  {X_test.shape}, Y_test shape:  {Y_test.shape}")

## 2. Train Primary Stacked LSTM Model

In [ ]:
from ml.train import build_lstm_model, train_model

model = build_lstm_model(input_shape=(LOOKBACK, len(FEATURES)), horizon=HORIZON, num_features=len(FEATURES))
model.summary()

In [ ]:
checkpoint_path = os.path.join(ROOT_DIR, "ml", "models", "vitals_lstm_notebook.keras")
trained_model, history = train_model(
    model,
    X_train,
    Y_train,
    X_val,
    Y_val,
    model_save_path=checkpoint_path,
    epochs=20,
    batch_size=128
)

## 3. Benchmark Evaluation against Clinical Baselines

In [ ]:
from ml.train import evaluate_baseline_last_value, evaluate_baseline_moving_avg, evaluate_model

res_last = evaluate_baseline_last_value(X_test, Y_test, scaler)
res_ma = evaluate_baseline_moving_avg(X_test, Y_test, scaler, window_len=15)
res_lstm = evaluate_model(trained_model, X_test, Y_test, scaler, model_name="LSTM")

summary_table = pd.DataFrame([
    {"Model": res_last["model_name"], "Overall MAE": res_last["overall_mae"], "Overall RMSE": res_last["overall_rmse"], "HR MAE": res_last["features"]["heart_rate"]["mae"], "SBP MAE": res_last["features"]["systolic_bp"]["mae"], "DBP MAE": res_last["features"]["diastolic_bp"]["mae"]},
    {"Model": res_ma["model_name"], "Overall MAE": res_ma["overall_mae"], "Overall RMSE": res_ma["overall_rmse"], "HR MAE": res_ma["features"]["heart_rate"]["mae"], "SBP MAE": res_ma["features"]["systolic_bp"]["mae"], "DBP MAE": res_ma["features"]["diastolic_bp"]["mae"]},
    {"Model": res_lstm["model_name"], "Overall MAE": res_lstm["overall_mae"], "Overall RMSE": res_lstm["overall_rmse"], "HR MAE": res_lstm["features"]["heart_rate"]["mae"], "SBP MAE": res_lstm["features"]["systolic_bp"]["mae"], "DBP MAE": res_lstm["features"]["diastolic_bp"]["mae"]},
])

summary_table

## 4. Monte Carlo Dropout Uncertainty Forecast (predict.py)

In [ ]:
from ml.predict import forecast

sample_patient = test_df["patient_id"].iloc[0]
sample_input = test_df[test_df["patient_id"] == sample_patient].iloc[:60]

pred_dict = forecast(sample_input, n_passes=30, confidence=0.90)
print("Predicted Next 12 Minutes Mean HR:", pred_dict["mean"]["heart_rate"])
print("90% Confidence Interval Lower:", pred_dict["lower"]["heart_rate"])
print("90% Confidence Interval Upper:", pred_dict["upper"]["heart_rate"])